# Lesson 18 Lab — Online Softmax and Fused Attention

**Puzzle:** When score tiling, running normalizers, and avoided materialization change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates score tiling, running normalizers, and avoided materialization and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

Fused attention tiles QK, updates an online Softmax normalizer, and immediately consumes probabilities in PV. The important invariant is that the full score matrix need not reside in HBM. Implementation details vary by backend, head dimension, causal mask, and generation.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["score tiling, running normalizers, and avoided materialization"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: optimized framework path described in the experiment.

FlashAttention is an algorithm family, not a promise that every SDPA call uses one named kernel.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 18
LESSON_TITLE = 'Online Softmax and Fused Attention'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260831
}


## 5. Freeze the experiment

**Experiment:** Compare PyTorch causal SDPA with explicitly materialized eager attention and calculate score-tensor bytes.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.026176000013947487,
  "secondary": 0.07769599929451942,
  "max_abs_error": 0.001953125,
  "passed": true,
  "details": {
    "materialized_score_bytes": 8388608,
    "sdpa_samples_ms": [
      0.03686400130391121,
      0.027424000203609467,
      0.025696000084280968,
      0.02598400041460991,
      0.026176000013947487,
      0.027424000203609467,
      0.02691200003027916,
      0.026079999282956123,
      0.024992000311613083,
      0.026399999856948853,
      0.024960000067949295,
      0.02579200081527233,
      0.025887999683618546,
      0.026623999699950218,
      0.02687999978661537
    ],
    "eager_samples_ms": [
      0.08684799820184708,
      0.07913599908351898,
      0.07625599950551987,
      0.07321599870920181,
      0.07276800274848938,
      0.07267200201749802,
      0.07327999919652939,
      0.08534400165081024,
      0.12624000012874603,
      0.14787200093269348
    ],
    "backend_claim": "PyTorch SDPA; internal kernel not asserted"
  }
}
P

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| SDPA median | 0.0262 ms |
| Materialized eager median | 0.0777 ms |
| Maximum absolute error | 1.953e-03 |
| Acceptance gate | true |


## 8. Explain without overclaiming

PyTorch SDPA took 0.0262 ms versus 0.0777 ms for materialized eager attention. Avoiding a 8.0 MiB score tensor is the algorithmic boundary; the internal SDPA kernel is not named.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Use the materialization model to explain scaling, but name the internal backend only when profiler evidence identifies it.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 18,
  "title": "Online Softmax and Fused Attention",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260831
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.026176000013947487,
    "secondary": 0.07769599929451942,
    "max_abs_error": 0.001953125,
    "passed": true,
    "details": {
      "materialized_score_bytes": 8388608,
      "sdpa_samples_ms": [
        0.03686400130391121,
        0.027424000203609467,
        0.025696000084280968,
        0.02598400041460991,
        0.026176000013947487,
        0.027424000203609467,
        0.02691200003027916,
        0.026079999282956123,
        0.024992000311613083,
        0.026399999856948853,
        0.024960000067949295,
        0.02579200081527233,
        0.025

## 10. Make the bounded decision

> Use the materialization model to explain scaling, but name the internal backend only when profiler evidence identifies it.

**Failure analysis:** FlashAttention is an algorithm family, not a promise that every SDPA call uses one named kernel.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
